# Modelling: multi-class service request routing

This notebook trains four classifiers to predict which city department
(`agency_responsible`) should handle each 311 request:

1. Logistic Regression
2. Decision Tree
3. Random Forest
4. Gradient Boosting

We use stratified splits, cross-validate, and compare accuracy and weighted F1.

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import cross_val_score, StratifiedKFold

sys.path.insert(0, '..')
from src.data_loader import load_or_fetch_data, preprocess_data, engineer_features
from src.model import (
    prepare_model_data,
    train_models,
    get_feature_importance,
    save_model,
)

DATA_DIR = str(Path('..') / 'data')
MODEL_DIR = str(Path('..') / 'models')
pd.set_option('display.max_columns', 40)
print('Setup complete.')

## 1. Prepare data

In [ ]:
raw = load_or_fetch_data(DATA_DIR, limit=100000)
df = preprocess_data(raw)
df = engineer_features(df)

X, y, label_encoders, feature_names = prepare_model_data(df)

print(f'Feature matrix: {X.shape}')
print(f'Target classes: {len(label_encoders["_target"].classes_)}')
print(f'Features: {feature_names}')

## 2. Stratified split and train all four classifiers

The `train_models` function handles the 80/20 split, scales features for
Logistic Regression, and trains all four models.

In [ ]:
trained_models, results, scaler, X_test, y_test = train_models(X, y, random_state=42)

results_df = pd.DataFrame(results).T.reset_index()
results_df.columns = ['Model', 'Accuracy', 'Weighted F1', 'Macro F1']
results_df = results_df.sort_values('Weighted F1', ascending=False)
results_df

In [ ]:
fig = px.bar(
    results_df.melt(id_vars='Model', value_vars=['Accuracy', 'Weighted F1', 'Macro F1']),
    x='Model', y='value', color='variable', barmode='group',
    title='Model comparison: accuracy and F1 scores',
    labels={'value': 'Score', 'variable': 'Metric'},
)
fig.update_layout(height=420)
fig.show()

## 3. Cross-validation

We run 5-fold stratified cross-validation on the full dataset to get
more robust accuracy estimates.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_models = {
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, n_jobs=-1)),
    'Decision Tree': DecisionTreeClassifier(max_depth=15, min_samples_split=10, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42),
}

cv_results = []
for name, model in cv_models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
    cv_results.append({
        'Model': name,
        'Mean accuracy': round(scores.mean(), 4),
        'Std': round(scores.std(), 4),
    })
    print(f'{name}: {scores.mean():.4f} (+/- {scores.std():.4f})')

cv_df = pd.DataFrame(cv_results)
cv_df

## 4. Compare accuracy and F1

Combining hold-out and cross-validation results to select the best model.

In [ ]:
merged = results_df.merge(cv_df[['Model', 'Mean accuracy']], on='Model', how='left')
merged = merged.rename(columns={'Mean accuracy': 'CV Accuracy'})
merged

## 5. Hyperparameter tuning (Gradient Boosting)

A quick grid search over learning rate and number of estimators for the
Gradient Boosting model.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [4, 6],
}

gb = GradientBoostingClassifier(random_state=42)
grid = GridSearchCV(gb, param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=0)
grid.fit(X, y)

print(f'Best params: {grid.best_params_}')
print(f'Best CV accuracy: {grid.best_score_:.4f}')

In [ ]:
# Results table
grid_results = pd.DataFrame(grid.cv_results_)
grid_results = grid_results[['param_n_estimators', 'param_learning_rate', 'param_max_depth', 'mean_test_score', 'std_test_score']]
grid_results = grid_results.sort_values('mean_test_score', ascending=False)
grid_results.head(8)

## 6. Feature importance

In [ ]:
rf_model = trained_models['Random Forest']
importance = get_feature_importance(rf_model, feature_names, 'Random Forest')

fig = px.bar(importance, x='Importance', y='Feature', orientation='h',
             title='Random Forest feature importance',
             color_discrete_sequence=['steelblue'])
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=400)
fig.show()

importance

## 7. Save best model

In [ ]:
best_name = results_df.iloc[0]['Model']
best_model = trained_models[best_name]
print(f'Best model: {best_name}')

save_model(best_model, scaler, label_encoders, feature_names, MODEL_DIR)
print(f'Model artifacts saved to {MODEL_DIR}')

## Summary

- All four classifiers were trained on the same 80/20 stratified split.
- Gradient Boosting and Random Forest typically outperform Logistic Regression and Decision Tree on weighted F1.
- Cross-validation confirms hold-out results are stable.
- Hyperparameter tuning via grid search yields a small improvement.
- `service_type_frequency` and `community_request_count` tend to be high-importance features.

Detailed error analysis follows in `04_evaluation.ipynb`.